In [ ]:
import requests,os
import time
from dotenv import load_dotenv

load_dotenv()

endpoint ="https://vandoc.cognitiveservices.azure.com/"
key = "5cUtNAfmEXwQv3zdUltq2asxl6U17zHB2ONtkxN5n0xhtrmlkxzhJQQJ99CDACNns7RXJ3w3AAALACOGS7ww"

url = endpoint + "formrecognizer/documentModels/prebuilt-invoice:analyze?api-version=2023-07-31"


headers = {
    "Ocp-Apim-Subscription-Key": key,
    "Content-Type": "image/jpeg"
}

# step 1: send invoice
with open("receipt.png", "rb") as f:
    data = f.read()

response = requests.post(url, headers=headers, data=data)

print("Step 1 Status:", response.status_code)
print("Headers:", response.headers)

# check if request accepted
if response.status_code == 202:

    result_url = response.headers["operation-location"]

    # wait for processing
    time.sleep(5)

    # step 2: get result
    result = requests.get(result_url,
                          headers={"Ocp-Apim-Subscription-Key": key})

    print("Final Result:")
    print(result.json())

else:
    print("Error:")
    print(response.text)





Step 1 Status: 202
Headers: {'Content-Length': '0', 'Operation-Location': 'https://kushagradoc.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-invoice/analyzeResults/6e7e27e2-22bb-4825-897e-1396d15ed60e?api-version=2023-07-31', 'x-envoy-upstream-service-time': '57', 'apim-request-id': '6e7e27e2-22bb-4825-897e-1396d15ed60e', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'Korea Central', 'Date': 'Sat, 11 Apr 2026 06:46:26 GMT'}
Final Result:
{'status': 'succeeded', 'createdDateTime': '2026-04-11T06:46:26Z', 'lastUpdatedDateTime': '2026-04-11T06:46:28Z', 'analyzeResult': {'apiVersion': '2023-07-31', 'modelId': 'prebuilt-invoice', 'stringIndexType': 'textElements', 'content': 'Company Name\nINVOICE\n[Street Address]\n[City, ST ZIP]\nDATE\n12/9/2019\nPhone: [000-000-0000]\nINVOICE #\n[123456]\nFax: [000-000-0000]\nCUSTOMER ID\n[123]\nWebsite: somedomain.com\nDUE DATE\n1/8/2020\nBILL TO\n[Na

In [4]:
result_json = result.json()

fields = result_json["analyzeResult"]["documents"][0]["fields"]

structured_output = {
    "VendorName": fields.get("VendorName", {}),
    "CustomerName": fields.get("CustomerName", {}),
    "InvoiceId": fields.get("InvoiceId", {}),
    "InvoiceDate": fields.get("InvoiceDate", {}),
    "DueDate": fields.get("DueDate", {}),
    "TotalAmount": fields.get("InvoiceTotal", {}),
    "SubTotal": fields.get("SubTotal", {}),
    "Tax": fields.get("TotalTax", {}),
    "Items": fields.get("Items", {})
}

import json
print(json.dumps(structured_output, indent=2))

{
  "VendorName": {},
  "CustomerName": {},
  "InvoiceId": {
    "type": "string",
    "valueString": "[123456]",
    "content": "[123456]",
    "boundingRegions": [
      {
        "pageNumber": 1,
        "polygon": [
          781,
          144,
          844,
          144,
          845,
          162,
          782,
          162
        ]
      }
    ],
    "confidence": 0.965,
    "spans": [
      {
        "offset": 100,
        "length": 8
      }
    ]
  },
  "InvoiceDate": {
    "type": "date",
    "valueDate": "2019-12-09",
    "content": "12/9/2019",
    "boundingRegions": [
      {
        "pageNumber": 1,
        "polygon": [
          777,
          123,
          851,
          122,
          851,
          138,
          777,
          138
        ]
      }
    ],
    "confidence": 0.967,
    "spans": [
      {
        "offset": 58,
        "length": 9
      }
    ]
  },
  "DueDate": {
    "type": "date",
    "valueDate": "2020-01-08",
    "content": "1/8/2020",
   

In [5]:
import json

result_json = result.json()
# print(result_json.analyzeResult.pages[0].words)
for word in result_json['analyzeResult']['pages'][0]['words']:
    print(word["content"])

# print(json.dumps(result_json, indent=2))

Company
Name
INVOICE
[Street
Address]
[City,
ST
ZIP]
DATE
12/9/2019
Phone:
[000-000-0000]
INVOICE
#
[123456]
Fax:
[000-000-0000]
CUSTOMER
ID
[123]
Website:
somedomain.com
DUE
DATE
1/8/2020
BILL
TO
[Name]
[Company
Name]
[Street
Address]
[City,
ST
ZIP]
[Phone]
DESCRIPTION
TAXED
AMOUNT
[Service
Fee]
230.00
[Labor:
5
hours
at
$75/hr]
375.00
[Parts]
X
345.00
Subtotal
950.00
Taxable
345.00
OTHER
COMMENTS
Tax
rate
6.250%
1.
Total
payment
due
in
30
days
Tax
due
21.56
2.
Please
include
the
invoice
number
on
your
check
Other
.
TOTAL
$
971.56
Make
all
checks
payable
to
[Your
Company
Name]
If
you
have
any
questions
about
this
invoice,
please
contact
[Name,
Phone
#,
E-mail]
Thank
You
For
Your
Business!
https://www.vertex42.com/ExcelTemplates/excel-invoice-template.html
Invoice
Template
@
2010-2019
by
Vertex42.com
